<a href="https://colab.research.google.com/github/hinaabbaskhan/BetterRest/blob/main/hairline_detection_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💇 Hairline Detection v3

---

## What problem are we solving?

Given a portrait photo, we want to find the **exact pixel row where the hair meets the forehead skin** — the hairline.

This is harder than it sounds:
- Hair color varies enormously (black, blonde, grey, dyed)
- Lighting changes how skin and hair look
- Hairstyles vary: side-swept bangs, center parts, receding hairlines, widow's peaks
- The face may be tilted or partially obscured

**Our solution:** combine two MediaPipe models — one for face geometry, one for hair pixels — and extract the hairline from where they intersect.

---

## Architecture overview

```
INPUT IMAGE
    │
    ├─ Step 1: MediaPipe Face Landmarker
    │          → tells us WHERE the forehead is
    │            (brow_y, x_left, x_right)
    │
    ├─ Step 2: MediaPipe Hair Segmenter
    │          → tells us WHICH pixels are hair
    │            (binary mask: 255=hair, 0=not hair)
    │
    ├─ Step 3: Per-column lowest hair pixel
    │          → for each x column in the forehead region,
    │            find the lowest hair pixel above the brow
    │            → raw contour: one y value per column
    │
    └─ Step 4: Smooth the contour
               → remove outliers, apply moving average
               → final hairline_y (single number)
               → smoothed curve (for visualization)
```

Each step is independent and inspectable.


---
## Why two models?

You might wonder: why not just use the hair mask alone?

**Hair mask alone fails** because the mask covers the entire hair region — top of head, sides, back. It has no concept of "forehead". Without knowing where the forehead is, you cannot find where hair *starts* at the scalp.

**Face landmarks alone fail** because MediaPipe Face Mesh only gives landmarks *up to the brow*. It does not have a landmark at the hairline itself — the forehead top is estimated, not measured.

**Together they work:**
- Face Mesh tells us the search region (between brow and ~65% of face height above brow)
- Hair mask tells us exactly which pixels are hair within that region
- The boundary between them, per column, is the hairline

---


## Step 1 — Face Landmarker: finding the forehead region

MediaPipe Face Landmarker gives us **478 facial landmarks** — precise x,y coordinates mapped onto the face.

We only use a small subset:


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# The landmark indices we actually use
landmarks_used = {
    'Left brow top':  [70, 63, 105, 66, 107],   # upper edge of left eyebrow
    'Right brow top': [336, 296, 334, 293, 300], # upper edge of right eyebrow
    'Left temple':    [234],                      # left side of face
    'Right temple':   [454],                      # right side of face
    'Chin tip':       [152],                      # used to compute face height
    'Nose tip':       [1],                        # used as center x reference
}

print("Landmark groups we use:")
for name, indices in landmarks_used.items():
    print(f"  {name:20s} → indices {indices}")

print("""
What we compute from them:
  brow_y  = min(all brow top y values)   → top of eyebrow = bottom of search zone
  x_left  = landmark[234].x             → left search boundary
  x_right = landmark[454].x             → right search boundary
  face_h  = chin_y - brow_y             → used to set search height limit
  nose_x  = landmark[1].x               → forehead center reference
""")


The result is a simple bounding box for the forehead:

```
x_left ─────────────────────────────── x_right
        ↑  search zone  (max 65% of face_h tall)
brow_y ─────────────────────────────────────────
```

**Why only these landmarks?** We deliberately use as little as possible from Face Mesh. More landmarks = more complexity = more things that can go wrong. We only need the box.

**Why use Face Mesh at all for this?** Alternatives like dlib fail on tilted or partially obscured faces. MediaPipe Face Landmarker is robust because it uses a neural network trained on diverse poses.

---


## Step 2 — Hair Segmenter: the hair mask

MediaPipe Hair Segmenter is a neural network trained to classify each pixel as either **hair** or **not hair**. It outputs a category mask where:
- `1` = hair
- `0` = everything else (skin, background, clothes)

We convert this to a binary image: `255` = hair, `0` = not hair.

### Why morphological cleanup?

The raw mask from the model has two problems:

| Problem | Example | Fix |
|---|---|---|
| Small holes inside hair region | A bright highlight in hair reads as "not hair" | `MORPH_CLOSE` (dilation then erosion) fills holes |
| Salt-and-pepper noise at edges | Stray pixels classified as hair near the face boundary | `MORPH_OPEN` (erosion then dilation) removes them |

```python
k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
mask    = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close)  # fill holes first
mask    = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k_open)   # then remove noise
```

Order matters: close before open, because opening after closing preserves the filled regions while still removing edge noise.

---


## Step 3 — Per-column lowest hair pixel

This is the core idea. Once we have:
- A bounding box for the forehead (from Step 1)
- A hair mask (from Step 2)

We scan **every x column** in the forehead region and ask:
> "What is the lowest (closest to brow) hair pixel in this column?"

That pixel is the hairline at that x position.


In [ ]:
# Pseudocode illustration of the per-column scan

def explain_per_column_scan():
    print("For each x column from x_left to x_right:")
    print()
    print("  col = hair_mask[y_top : brow_y, x]")
    print("       ↑ vertical slice from search ceiling down to brow")
    print()
    print("  if col has too little hair → skip (likely forehead gap or center part)")
    print()
    print("  hair_idxs = np.where(col > 128)[0]")
    print("  lowest_hair = hair_idxs.max()      ← max index = lowest row")
    print("  hairline_y_at_x = y_top + lowest_hair")
    print()
    print("Result: a list of (x, y) points forming a raw contour")
    print("        one y value per column = the bottom edge of hair at that column")

explain_per_column_scan()


### Why "lowest" and not "topmost"?

The hair mask covers the **entire hair region** — from scalp down into the face if hair is long. We want the **bottom boundary** of hair in the forehead zone, which is where hair meets skin.

- Topmost hair pixel → top of head (not what we want)
- Lowest hair pixel within the forehead zone → hairline ✓

### The margin trim

We shrink the x search range by `margin_frac` (default 5%) on each side:

```python
margin = int((x_right - x_left) * margin_frac)
x0     = x_left  + margin
x1     = x_right - margin
```

This avoids the temple edges where sideburn/ear hair would pull the contour down to the wrong place.

### The minimum hair fraction filter

```python
if hair_px / total < min_hair_col_frac:
    continue  # skip this column
```

Columns in the center of a center-parted hairstyle may have very little hair — they are forehead skin all the way up. Skipping them avoids a dip in the contour at the parting.

---


## Step 4 — Smooth the contour

The raw per-column contour has two problems:

1. **Outliers** — columns at the edge of sideburns or near the temples can produce y values far below the true hairline
2. **Noise** — individual column detections jitter up and down by a few pixels

We fix both:

### Outlier removal

```python
med  = np.median(ys)
std  = np.std(ys)
keep = np.abs(ys - med) <= outlier_std * std   # default: 1.5 std
```

Points more than 1.5 standard deviations from the median are dropped. This handles the case where a few columns detect sideburn hair that is much lower than the actual hairline.

### Moving average smoothing

```python
kernel  = np.ones(win) / win     # box filter
ys_pad  = np.pad(ys, win//2, mode='edge')   # pad to avoid edge shrinkage
ys_sm   = np.convolve(ys_pad, kernel, mode='valid')[:len(ys)]
```

A box filter (simple moving average) over `smooth_win` columns (default 31 pixels) removes pixel-level jitter while preserving real shape like widow's peaks or curved hairlines.

**Why not Gaussian?** Box filter is simpler, fast, and works well here because the underlying contour is already smooth after outlier removal.

### Single scalar output

```python
n          = len(ys_sm)
lo, hi     = n // 3, (2 * n) // 3
center     = ys_sm[lo:hi]
hairline_y = int(np.median(center))
```

We take the median of the **center third** of the smoothed contour — ignoring the left and right edges where temple hair might still bias the result downward.

---


## Parameter reference

All parameters have sensible defaults but can be tuned per use case using the sliders in Cell 13 of the main notebook.

| Parameter | Default | What it controls | Increase if... | Decrease if... |
|---|---|---|---|---|
| `margin_frac` | 0.05 | Temple edge trim (fraction of face width) | Sideburn/ear hair pulls contour down at edges | Hairline is cut off at sides |
| `max_up_frac` | 0.65 | How far above brow to search (fraction of face height) | Hairline not found (high forehead) | Line goes too far into hair above scalp |
| `min_hair_col_frac` | 0.10 | Minimum hair fraction for a column to count | Center-part gap causes a dip in contour | Too many columns skipped (sparse hair) |
| `smooth_win` | 31 | Moving average window width in pixels | Contour is too jagged | Real hairline shape (widow's peak) is over-smoothed |
| `outlier_std` | 1.5 | How aggressively to remove outlier columns | Sideburn columns still appear | Valid edge columns are being dropped |

---


## Known failure modes

| Situation | What happens | Workaround |
|---|---|---|
| **No face detected** | Step 1 returns None, pipeline stops | Ensure face is clearly visible and roughly frontal |
| **Very dark hair + dark skin** | Hair mask boundary may be fuzzy | Increase `smooth_win` to 51+, reduce `outlier_std` to 1.0 |
| **Blonde / light hair** | Hair mask may miss parts of hair region | No change needed — model handles this reasonably |
| **Head covering (hijab, hat)** | Hair mask returns little/no hair | Pipeline returns `success=False` gracefully |
| **Heavy side-swept bangs** | Contour dips on one side | Increase `margin_frac` to 0.10–0.15 to avoid bang region |
| **Receding hairline** | Center columns have no hair → skipped | Decrease `min_hair_col_frac` to 0.05 |
| **Very tilted head (>45°)** | Face Mesh may misplace brow_y | Pipeline detects this via sanity check, returns error |

---


## Output reference

`detect_hairline_v3()` returns a dict with everything:

```python
result = {
    # Top-level answer
    'success':    True/False,
    'hairline_y': 243,          # final pixel row — USE THIS
    'error':      '',           # set if success=False

    # Step 1 output
    'bounds': {
        'brow_y':   340,        # top of eyebrow
        'face_h':   412,        # chin_y - brow_y
        'x_left':   198,        # left temple x
        'x_right':  482,        # right temple x
        'nose_x':   340,        # center x
        'img_h':    800,
        'img_w':    600,
    },

    # Step 2 output
    'hair_mask': np.ndarray,    # H×W uint8, 255=hair

    # Step 3 output
    'contour': {
        'xs':     [...],        # x positions with valid detections
        'ys_raw': [...],        # raw hairline y per column
        'x0': 208, 'x1': 472,  # actual search bounds after margin trim
        'y_top': 73, 'y_bot': 340,
    },

    # Step 4 output
    'smoothed': {
        'xs_smooth': np.ndarray,  # x after outlier removal
        'ys_smooth': np.ndarray,  # smoothed y values (the contour curve)
        'hairline_y': 243,        # same as top-level hairline_y
        'success': True,
    },
}
```

For production use you only need `result['hairline_y']`.
All other fields are there for debugging and visualization.

---


## Integrating into the Lambda project

The pipeline uses only these libraries:
- `mediapipe` (models downloaded on first run, cached to disk)
- `opencv-python`
- `numpy`

### Initialization (once at cold start)

```python
# In your Lambda handler module — top level, outside the handler function
face_landmarker = build_face_landmarker()
hair_segmenter  = build_hair_segmenter()
```

Putting initialization at module level means it runs once on cold start and is reused across warm invocations — critical for Lambda performance.

### Per-request call

```python
def handler(event, context):
    img_bytes = base64.b64decode(event['image'])
    img_array = np.frombuffer(img_bytes, np.uint8)
    img_bgr   = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    result = detect_hairline_v3(img_bgr)

    if not result['success']:
        return {'statusCode': 400, 'error': result['error']}

    return {'statusCode': 200, 'hairline_y': result['hairline_y']}
```

### Model file handling in Docker

The model files (`face_landmarker.task`, `hair_segmenter.tflite`) should be **baked into the Docker image** rather than downloaded at runtime:

```dockerfile
# In your Dockerfile
COPY models/face_landmarker.task   /app/models/
COPY models/hair_segmenter.tflite  /app/models/
```

Then update `FACE_MODEL_PATH` and `HAIR_MODEL_PATH` to `/app/models/...`.

This avoids cold-start network latency and makes the container fully self-contained.

---


## Visual summary of the pipeline

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.4)

ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1])
ax2 = fig.add_subplot(gs[2])
ax3 = fig.add_subplot(gs[3])

# ── Step 1: forehead box ──────────────────────────────────────
ax0.set_xlim(0, 10); ax0.set_ylim(0, 14); ax0.invert_yaxis()
ax0.set_aspect('equal')
# Face outline
face = plt.Polygon([[2,1],[8,1],[9,9],[5,13],[1,9]], fill=False,
                    edgecolor='#e8c4a0', lw=2)
ax0.add_patch(face)
# Brow line
ax0.axhline(5, xmin=0.15, xmax=0.85, color='#888', lw=1.5, ls='--')
ax0.text(5, 4.5, 'brow_y', ha='center', fontsize=8, color='#888')
# Search box
rect = plt.Rectangle((2, 1.5), 6, 3.5, fill=True,
                       facecolor='#00e67622', edgecolor='#00e676', lw=2)
ax0.add_patch(rect)
ax0.text(5, 0.8, 'Search zone', ha='center', fontsize=9, color='#00e676')
ax0.text(5, 12, 'Step 1: Face Mesh
Forehead bounds', ha='center',
         fontsize=10, fontweight='bold', va='bottom')
ax0.axis('off')

# ── Step 2: hair mask ─────────────────────────────────────────
ax1.set_xlim(0, 10); ax1.set_ylim(0, 14); ax1.invert_yaxis()
ax1.set_aspect('equal')
face2 = plt.Polygon([[2,1],[8,1],[9,9],[5,13],[1,9]], fill=False,
                     edgecolor='#e8c4a0', lw=2)
ax1.add_patch(face2)
hair = plt.Polygon([[1,9],[2,5],[3,3],[5,2],[7,3],[8,5],[9,9],[5,13]],
                    fill=True, facecolor='#3a3a8044', edgecolor='#5555cc', lw=2)
ax1.add_patch(hair)
ax1.text(5, 7, 'HAIR
(mask=255)', ha='center', fontsize=9, color='#5555cc')
ax1.text(5, 10.5, 'SKIN
(mask=0)',   ha='center', fontsize=9, color='#cc8844')
ax1.text(5, 12, 'Step 2: Hair Mask
Segmentation', ha='center',
         fontsize=10, fontweight='bold', va='bottom')
ax1.axis('off')

# ── Step 3: per-column scan ───────────────────────────────────
ax2.set_xlim(0, 10); ax2.set_ylim(0, 14); ax2.invert_yaxis()
ax2.set_aspect('equal')
face3 = plt.Polygon([[2,1],[8,1],[9,9],[5,13],[1,9]], fill=False,
                     edgecolor='#e8c4a0', lw=2)
ax2.add_patch(face3)
hair3 = plt.Polygon([[1,9],[2,5],[3,3],[5,2],[7,3],[8,5],[9,9],[5,13]],
                     fill=True, facecolor='#3a3a8044', edgecolor='#5555cc', lw=2)
ax2.add_patch(hair3)
# Show column scan arrows
for xi, yi in [(3,4.2),(4,3.3),(5,2.8),(6,3.3),(7,4.2)]:
    ax2.annotate('', xy=(xi, yi), xytext=(xi, 1.5),
                 arrowprops=dict(arrowstyle='->', color='#ff9800', lw=1.5))
    ax2.plot(xi, yi, 'o', color='#ff9800', markersize=5)
ax2.text(5, 12, 'Step 3: Per-column
Lowest hair pixel', ha='center',
         fontsize=10, fontweight='bold', va='bottom')
ax2.axis('off')

# ── Step 4: smooth contour ────────────────────────────────────
ax3.set_xlim(0, 10); ax3.set_ylim(0, 14); ax3.invert_yaxis()
ax3.set_aspect('equal')
face4 = plt.Polygon([[2,1],[8,1],[9,9],[5,13],[1,9]], fill=False,
                     edgecolor='#e8c4a0', lw=2)
ax3.add_patch(face4)
hair4 = plt.Polygon([[1,9],[2,5],[3,3],[5,2],[7,3],[8,5],[9,9],[5,13]],
                     fill=True, facecolor='#3a3a8044', edgecolor='#5555cc', lw=2)
ax3.add_patch(hair4)
# Raw contour (jagged)
xs_raw = [3,  3.5, 4,  4.5, 5,  5.5, 6,  6.5, 7]
ys_raw = [4.4,3.7, 3.5,3.1, 2.9,3.2, 3.5,3.9, 4.3]
ax3.plot(xs_raw, ys_raw, 'o--', color='#ffcc00', markersize=3,
         lw=1, alpha=0.6, label='raw')
# Smoothed (clean)
xs_sm = np.linspace(3, 7, 50)
ys_sm = 2.9 + 0.35 * (xs_sm - 5)**2
ax3.plot(xs_sm, ys_sm, '-', color='#00e676', lw=2.5, label='smoothed')
# hairline_y
ax3.axhline(3.1, color='#00e676', lw=1, ls=':', alpha=0.5)
ax3.text(8.2, 3.1, 'hairline_y', fontsize=7.5, color='#00e676', va='center')
ax3.legend(fontsize=8, loc='lower right')
ax3.text(5, 12, 'Step 4: Smooth
Contour', ha='center',
         fontsize=10, fontweight='bold', va='bottom')
ax3.axis('off')

plt.suptitle('Hairline Detection v3 — Pipeline Overview', fontsize=13, y=1.02)
plt.savefig('/tmp/pipeline_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('Pipeline diagram rendered.')